# Paper Baseline WMDP Full No-Defense Evaluation

This notebook runs the full paper baseline/control path: direct target-model evaluation without PoRT, without classifier gating, and without corruption hooks. It evaluates WMDP original prompts plus the paper-style adversarial variants available in this repo (`noise_prefix` and `composite`) across all WMDP domains.

This is the full-dataset follow-up to `10_kaggle_paper_baseline_wmdp_smoke_test.ipynb`. It is intentionally not the PoRT main method; it establishes the no-defense baseline before running a paper-faithful PoRT smoke/full pipeline.


In [1]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()

def has_project_layout(path):
    path = Path(path)
    return (path / 'llm-unlearn-eco' / 'eco').exists() and (path / 'dataset').exists()

def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()

PROJECT_ROOT = clone_or_use_project()
ECO_ROOT = PROJECT_ROOT / 'llm-unlearn-eco'
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(ECO_ROOT) not in sys.path:
    sys.path.insert(0, str(ECO_ROOT))

commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')

Cloning https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git into /kaggle/working/PoRT_LLM_Unlearning-Experiment


Cloning into '/kaggle/working/PoRT_LLM_Unlearning-Experiment'...


Project root: /kaggle/working/PoRT_LLM_Unlearning-Experiment
Commit: 0e85d41697401d9098f601346b6d52b7072be774


In [2]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'tabulate': 'tabulate',
    'tqdm': 'tqdm',
    'transformers': 'transformers>=4.38.0',
    'accelerate': 'accelerate>=0.26.0',
    'yaml': 'pyyaml',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')

Required packages are already available.


In [3]:
import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('This smoke test is intended for a Kaggle GPU runtime.')
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f'GPU {idx}: {props.name}, VRAM={props.total_memory / 1024**3:.2f} GiB')

Python: 3.12.13
Torch: 2.10.0+cu128
CUDA available: True
GPU 0: Tesla T4, VRAM=14.56 GiB
GPU 1: Tesla T4, VRAM=14.56 GiB


## Runtime Config

Defaults run the full dataset for `original`, `noise_prefix`, and `composite` over `bio`, `chem`, and `cyber`. Set `PORT_WMDP_SAMPLE_SIZE` to a positive integer only when you want a smaller debug run; leave it unset for full evaluation.


In [4]:
def optional_int_env(name):
    value = os.environ.get(name)
    if value is None:
        return None
    value = value.strip()
    if value.lower() in {'', 'none', 'null', 'full', 'all', '-1'}:
        return None
    parsed = int(value)
    if parsed <= 0:
        return None
    return parsed

TARGET_CONFIG_NAME = os.environ.get('PORT_TARGET_CONFIG_NAME', 'phi-1_5')
TARGET_HF_NAME = os.environ.get('PORT_TARGET_HF_NAME')
TARGET_MODEL_PATH = os.environ.get('PORT_TARGET_MODEL_PATH')
ATTN_IMPLEMENTATION = os.environ.get('PORT_ATTN_IMPLEMENTATION', 'eager')
TORCH_DTYPE = os.environ.get('PORT_TORCH_DTYPE', 'float16')
BATCH_SIZE = int(os.environ.get('PORT_WMDP_BATCH_SIZE', '1'))
SAMPLE_SIZE = optional_int_env('PORT_WMDP_SAMPLE_SIZE')
SEED = int(os.environ.get('PORT_SEED', '0'))

variant_env = os.environ.get('PORT_WMDP_BASELINE_VARIANTS', 'original,noise_prefix,composite')
EVAL_VARIANTS = [item.strip() for item in variant_env.split(',') if item.strip()]
domain_env = os.environ.get('PORT_WMDP_DOMAINS', 'bio,chem,cyber')
EVAL_DOMAINS = [item.strip() for item in domain_env.split(',') if item.strip()]

RUN_NAME = os.environ.get(
    'PORT_RUN_NAME',
    f'paper_baseline_wmdp_full_{TARGET_CONFIG_NAME}',
)
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT / 'results'
RUN_DIR = OUTPUT_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Run name:', RUN_NAME)
print('Variants:', EVAL_VARIANTS)
print('Domains:', EVAL_DOMAINS)
print('Sample size per domain/variant:', 'FULL' if SAMPLE_SIZE is None else SAMPLE_SIZE)
print('Batch size:', BATCH_SIZE)


Run name: paper_baseline_wmdp_full_phi-1_5
Variants: ['original', 'noise_prefix', 'composite']
Domains: ['bio', 'chem', 'cyber']
Sample size per domain/variant: FULL
Batch size: 1


## Load Target Model

In [5]:
import time
import yaml
from types import SimpleNamespace

from eco.model import HFModel
from eco.paths import MODEL_CONFIG_DIR
from eco.utils import delete_model, load_yaml, seed_everything

seed_everything(SEED)

def sanitize_name(value):
    import re
    return re.sub(r'[^A-Za-z0-9_.=-]+', '_', str(value)).strip('_')

def load_runtime_model_config():
    base_config_path = MODEL_CONFIG_DIR / f'{TARGET_CONFIG_NAME}.yaml'
    if not base_config_path.exists():
        raise FileNotFoundError(f'Model config not found: {base_config_path}')

    runtime_config = load_yaml(base_config_path)
    if TARGET_HF_NAME:
        runtime_config['hf_name'] = TARGET_HF_NAME
    if TORCH_DTYPE:
        runtime_config['torch_dtype'] = TORCH_DTYPE
    if ATTN_IMPLEMENTATION:
        runtime_config['attn_implementation'] = ATTN_IMPLEMENTATION

    runtime_config_dir = RUN_DIR / 'model_config'
    runtime_config_dir.mkdir(parents=True, exist_ok=True)
    runtime_model_name = sanitize_name(f'{TARGET_CONFIG_NAME}_runtime_paper_baseline_wmdp')
    runtime_config_path = runtime_config_dir / f'{runtime_model_name}.yaml'
    with runtime_config_path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(runtime_config, f, sort_keys=False)
    return runtime_model_name, runtime_config, runtime_config_path

runtime_model_name, runtime_config, runtime_config_path = load_runtime_model_config()
print(json.dumps({
    'runtime_model_name': runtime_model_name,
    'runtime_config_path': str(runtime_config_path),
    'target_hf_name': runtime_config.get('hf_name'),
    'torch_dtype': runtime_config.get('torch_dtype'),
    'attn_implementation': runtime_config.get('attn_implementation'),
}, indent=2))

started = time.perf_counter()
model = HFModel(
    model_name=runtime_model_name,
    model_path=TARGET_MODEL_PATH,
    config_path=str(runtime_config_path.parent),
)
if model.tokenizer.pad_token_id is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
model.tokenizer.padding_side = 'right'
model_load_seconds = time.perf_counter() - started
print(f'Model loaded in {model_load_seconds:.2f}s')

{
  "runtime_model_name": "phi-1_5_runtime_paper_baseline_wmdp",
  "runtime_config_path": "/kaggle/working/paper_baseline_wmdp_full_phi-1_5/model_config/phi-1_5_runtime_paper_baseline_wmdp.yaml",
  "target_hf_name": "microsoft/phi-1_5",
  "torch_dtype": "float16",
  "attn_implementation": "eager"
}


config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.84G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Model loaded in 17.69s


## Build WMDP Baseline Datasets

`original` uses the raw `question` column and repo prompt formatting. `noise_prefix` and `composite` use the preformatted `full_question` column so the adversarial prompt is actually evaluated.


In [6]:
from eco.dataset import WMDPBio, WMDPChem, WMDPCyber

DOMAIN_CLASSES = {
    'bio': WMDPBio,
    'chem': WMDPChem,
    'cyber': WMDPCyber,
}

VARIANT_PATHS = {
    'original': None,
    'noise_prefix': PROJECT_ROOT / 'dataset' / 'WMDP' / 'noise_prefix',
    'composite': PROJECT_ROOT / 'dataset' / 'WMDP' / 'composite',
}

def make_wmdp_module(variant, domain, sample_size):
    if domain not in DOMAIN_CLASSES:
        raise ValueError(f'Unknown WMDP domain: {domain}')
    dataset_cls = DOMAIN_CLASSES[domain]
    if variant == 'original':
        module = dataset_cls(sample_size=sample_size)
    else:
        root = VARIANT_PATHS.get(variant)
        if root is None:
            raise ValueError(f'Unknown WMDP variant: {variant}')
        dataset_path = root / domain
        if not dataset_path.exists():
            raise FileNotFoundError(f'Dataset path not found for {variant}/{domain}: {dataset_path}')
        module = dataset_cls(
            dataset_path=dataset_path,
            question_key='full_question',
            question_is_formatted=True,
        )
        module.download()
        if sample_size is not None and sample_size > 0:
            for split_name in list(module.dataset.keys()):
                split = module.dataset[split_name]
                module.dataset[split_name] = split.select(range(min(sample_size, len(split))))
    return module

DATA_MODULES = []
for variant in EVAL_VARIANTS:
    for domain in EVAL_DOMAINS:
        module = make_wmdp_module(variant, domain, SAMPLE_SIZE)
        DATA_MODULES.append({'variant': variant, 'domain': domain, 'module': module})
        if module.dataset is None:
            module.download()
        print(f'{variant}/{domain}: rows={len(module.dataset["test"])} question_key={module.question_key} formatted={module.question_is_formatted}')

expected_rows = sum(len(item['module'].dataset['test']) for item in DATA_MODULES)
print('Expected total rows:', expected_rows)

Generating train split: 0 examples [00:00, ? examples/s]

original/bio: rows=1273 question_key=question formatted=False


Generating train split: 0 examples [00:00, ? examples/s]

original/chem: rows=408 question_key=question formatted=False


Generating train split: 0 examples [00:00, ? examples/s]

original/cyber: rows=1987 question_key=question formatted=False
noise_prefix/bio: rows=1273 question_key=full_question formatted=True
noise_prefix/chem: rows=408 question_key=full_question formatted=True
noise_prefix/cyber: rows=1987 question_key=full_question formatted=True
composite/bio: rows=1273 question_key=full_question formatted=True
composite/chem: rows=408 question_key=full_question formatted=True
composite/cyber: rows=1987 question_key=full_question formatted=True
Expected total rows: 11004


## Run Full No-PoRT Baseline Evaluation

Each `variant/domain` job writes a per-job CSV and updates partial artifacts before moving to the next job. This keeps the run inspectable if a Kaggle session stops before all nine jobs finish.


In [7]:
import pandas as pd
from eco.evaluator import ChoiceByTopLogit
from eco.inference import EvaluationEngine

CHOICE_LABELS = ['A', 'B', 'C', 'D']

run_config = {
    'purpose': 'paper_baseline_no_port_wmdp_full',
    'project_root': str(PROJECT_ROOT),
    'commit': commit_sha,
    'model_name': TARGET_CONFIG_NAME,
    'target_hf_name': runtime_config.get('hf_name'),
    'model_path': TARGET_MODEL_PATH,
    'runtime_model_name': runtime_model_name,
    'runtime_config_path': str(runtime_config_path),
    'sample_size': SAMPLE_SIZE,
    'batch_size': BATCH_SIZE,
    'variants': EVAL_VARIANTS,
    'domains': EVAL_DOMAINS,
    'expected_rows': int(expected_rows),
    'run_name': RUN_NAME,
    'run_dir': str(RUN_DIR),
    'variant_prompt_source': {
        'original': 'question formatted by WMDP.format_prompt',
        'noise_prefix': 'full_question preformatted adversarial prompt',
        'composite': 'full_question preformatted adversarial prompt',
    },
}
(RUN_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2, default=str), encoding='utf-8')

def extract_predictions(outputs, variant, domain, data_module):
    predictions = []
    result_name, batches = list(outputs[0].items())[0]
    row_idx = 0
    for batch in batches:
        for correct, predicted in zip(batch['correct'], batch['predicted']):
            correct = int(correct)
            predicted = int(predicted)
            predictions.append({
                'variant': variant,
                'domain': domain,
                'dataset': data_module.name,
                'result_name': result_name,
                'row_index': row_idx,
                'correct_index': correct,
                'predicted_index': predicted,
                'correct_label': CHOICE_LABELS[correct] if 0 <= correct < len(CHOICE_LABELS) else str(correct),
                'predicted_label': CHOICE_LABELS[predicted] if 0 <= predicted < len(CHOICE_LABELS) else str(predicted),
                'is_correct': correct == predicted,
            })
            row_idx += 1
    return predictions

def write_partial_artifacts():
    partial_predictions = pd.DataFrame(all_predictions)
    if len(partial_predictions):
        partial_predictions.to_csv(RUN_DIR / 'predictions_partial.csv', index=False)
        partial_summary_by_variant_domain = (
            partial_predictions.groupby(['variant', 'domain'])['is_correct']
            .agg(['count', 'mean'])
            .reset_index()
            .rename(columns={'mean': 'accuracy'})
        )
        partial_summary_by_variant = (
            partial_predictions.groupby('variant')['is_correct']
            .agg(['count', 'mean'])
            .reset_index()
            .rename(columns={'mean': 'accuracy'})
        )
        partial_summary_overall = pd.DataFrame({
            'count': [int(len(partial_predictions))],
            'accuracy': [float(partial_predictions['is_correct'].mean())],
        })
        partial_summary_by_variant_domain.to_csv(RUN_DIR / 'summary_by_variant_domain_partial.csv', index=False)
        partial_summary_by_variant.to_csv(RUN_DIR / 'summary_by_variant_partial.csv', index=False)
        partial_summary_overall.to_csv(RUN_DIR / 'summary_overall_partial.csv', index=False)

    partial_payload = {
        'run_config': run_config,
        'model_load_seconds': model_load_seconds,
        'timings_seconds': timings,
        'completed_jobs': completed_jobs,
        'engine_summaries': engine_summaries,
        'expected_rows': int(expected_rows),
        'total_rows_so_far': int(len(partial_predictions)),
    }
    (RUN_DIR / 'summary_partial.json').write_text(json.dumps(partial_payload, indent=2, default=str), encoding='utf-8')
    pd.DataFrame(completed_jobs).to_csv(RUN_DIR / 'completed_jobs_partial.csv', index=False)

all_predictions = []
engine_summaries = []
completed_jobs = []
timings = {'model_load': model_load_seconds}

for item in DATA_MODULES:
    variant = item['variant']
    domain = item['domain']
    module = item['module']
    label = f'{variant}/{domain}'
    print()
    print('=' * 80)
    print(f'Running full no-PoRT baseline on {label}')
    started = time.perf_counter()
    engine = EvaluationEngine(
        model=model,
        tokenizer=model.tokenizer,
        data_module=module,
        subset_names=['test'],
        evaluator=ChoiceByTopLogit(save_logits=False),
        batch_size=BATCH_SIZE,
    )
    engine.inference()
    summary_stats, outputs = engine.summary()
    elapsed = time.perf_counter() - started
    predictions = extract_predictions(outputs, variant, domain, module)
    all_predictions.extend(predictions)
    job = {
        'variant': variant,
        'domain': domain,
        'dataset': module.name,
        'rows': int(len(predictions)),
        'seconds': float(elapsed),
        'summary': summary_stats,
    }
    completed_jobs.append(job)
    engine_summaries.append(job)
    timings[label] = elapsed
    pd.DataFrame(predictions).to_csv(RUN_DIR / f'predictions_{variant}_{domain}.csv', index=False)
    write_partial_artifacts()
    print(f'{label}: rows={len(predictions)}, seconds={elapsed:.2f}, summary={summary_stats}')

predictions_df = pd.DataFrame(all_predictions)
summary_by_variant_domain = (
    predictions_df.groupby(['variant', 'domain'])['is_correct']
    .agg(['count', 'mean'])
    .reset_index()
    .rename(columns={'mean': 'accuracy'})
)
summary_by_variant = (
    predictions_df.groupby('variant')['is_correct']
    .agg(['count', 'mean'])
    .reset_index()
    .rename(columns={'mean': 'accuracy'})
)
summary_overall = pd.DataFrame({
    'count': [int(len(predictions_df))],
    'accuracy': [float(predictions_df['is_correct'].mean())],
})

predictions_df.to_csv(RUN_DIR / 'predictions.csv', index=False)
summary_by_variant_domain.to_csv(RUN_DIR / 'summary_by_variant_domain.csv', index=False)
summary_by_variant.to_csv(RUN_DIR / 'summary_by_variant.csv', index=False)
summary_overall.to_csv(RUN_DIR / 'summary_overall.csv', index=False)
pd.DataFrame(completed_jobs).to_csv(RUN_DIR / 'completed_jobs.csv', index=False)

summary_payload = {
    'run_config': run_config,
    'model_load_seconds': model_load_seconds,
    'timings_seconds': timings,
    'completed_jobs': completed_jobs,
    'engine_summaries': engine_summaries,
    'summary_by_variant_domain': summary_by_variant_domain.to_dict(orient='records'),
    'summary_by_variant': summary_by_variant.to_dict(orient='records'),
    'summary_overall': summary_overall.to_dict(orient='records'),
    'expected_rows': int(expected_rows),
    'total_rows': int(len(predictions_df)),
}
(RUN_DIR / 'summary.json').write_text(json.dumps(summary_payload, indent=2, default=str), encoding='utf-8')

print()
print('Summary by variant/domain:')
print(summary_by_variant_domain.to_string(index=False))
print()
print('Summary by variant:')
print(summary_by_variant.to_string(index=False))
print()
print('Overall:')
print(summary_overall.to_string(index=False))
print(f'Wrote artifacts to: {RUN_DIR}')



Running full no-PoRT baseline on original/bio


Map:   0%|          | 0/1273 [00:00<?, ? examples/s]

Map:   0%|          | 0/1273 [00:00<?, ? examples/s]

Evaluating choice_by_top_logit of wmdp-bio on test: 100%|██████████| 1273/1273 [01:01<00:00, 20.59it/s]


{'wmdp-bio_test_choice_by_top_logit': 0.5239591516103692}
original/bio: rows=1273, seconds=62.51, summary=[{'wmdp-bio_test_choice_by_top_logit': 0.5239591516103692}]

Running full no-PoRT baseline on original/chem


...

{'wmdp-chem_test_choice_by_top_logit': 0.24019607843137256}
composite/chem: rows=408, seconds=20.23, summary=[{'wmdp-chem_test_choice_by_top_logit': 0.24019607843137256}]

Running full no-PoRT baseline on composite/cyber


Map:   0%|          | 0/1987 [00:00<?, ? examples/s]

Map:   0%|          | 0/1987 [00:00<?, ? examples/s]

Evaluating choice_by_top_logit of wmdp-cyber on test: 100%|██████████| 1987/1987 [04:29<00:00,  7.37it/s]


{'wmdp-cyber_test_choice_by_top_logit': 0.2622043281328636}
composite/cyber: rows=1987, seconds=270.50, summary=[{'wmdp-cyber_test_choice_by_top_logit': 0.2622043281328636}]

Summary by variant/domain:
     variant domain  count  accuracy
   composite    bio   1273  0.340141
   composite   chem    408  0.240196
   composite  cyber   1987  0.262204
noise_prefix    bio   1273  0.483111
noise_prefix   chem    408  0.323529
noise_prefix  cyber   1987  0.309512
    original    bio   1273  0.523959
    original   chem    408  0.335784
    original  cyber   1987  0.324107

Summary by variant:
     variant  count  accuracy
   composite   3668  0.286805
noise_prefix   3668  0.371320
    original   3668  0.394766

Overall:
 count  accuracy
 11004  0.350963
Wrote artifacts to: /kaggle/working/paper_baseline_wmdp_full_phi-1_5


## Verify Full Artifacts

This final cell fails if the run did not produce the expected row count or required artifacts. With default variants/domains and full sample size, expected total rows should be `11004` (`3668` rows per variant).


In [8]:
required_files = [
    'run_config.json',
    'summary.json',
    'predictions.csv',
    'summary_by_variant_domain.csv',
    'summary_by_variant.csv',
    'summary_overall.csv',
    'completed_jobs.csv',
    'predictions_partial.csv',
    'summary_partial.json',
]
missing_files = [name for name in required_files if not (RUN_DIR / name).exists()]
if missing_files:
    raise RuntimeError(f'Missing expected artifacts: {missing_files}')

if len(predictions_df) != expected_rows:
    raise RuntimeError(f'Expected {expected_rows} predictions, got {len(predictions_df)}')
if predictions_df['predicted_index'].isna().any():
    raise RuntimeError('Some predictions are missing predicted_index')
if predictions_df['correct_index'].isna().any():
    raise RuntimeError('Some predictions are missing correct_index')

if SAMPLE_SIZE is None and set(EVAL_VARIANTS) == {'original', 'noise_prefix', 'composite'} and set(EVAL_DOMAINS) == {'bio', 'chem', 'cyber'}:
    if expected_rows != 11004:
        raise RuntimeError(f'Default full paper baseline should have 11004 rows, got {expected_rows}')

print('PAPER BASELINE WMDP FULL NO-DEFENSE EVALUATION COMPLETED')
print('Rows:', len(predictions_df))
print('Artifacts:', RUN_DIR)


PAPER BASELINE WMDP FULL NO-DEFENSE EVALUATION COMPLETED
Rows: 11004
Artifacts: /kaggle/working/paper_baseline_wmdp_full_phi-1_5
